# E-Commerce Profit Leakage & Customer Value Analytics

**Objective.** E-commerce businesses generate substantial revenue but lose profit through excessive discounts, product returns, shipping costs, failed deliveries, payment fees, and low-margin products. This project analyses transaction, customer, product, discount, return, and operational data to identify major sources of profit leakage, measure customer profitability, and uncover actionable patterns.

**Dataset:** Global E-Commerce Dataset (1M+ records, 2024–2026) — https://www.kaggle.com/datasets/akrambelha/global-e-commerce-dataset-1m-records-20242026

**Technologies:** Python, Pandas, NumPy, Plotly, Dash (dashboard in `app.py`), pycountry. All values in USD; every metric below is computed from the dataset — nothing is hardcoded.

In [1]:
import pandas as pd
import plotly.io as pio
from utils.data_processing import load_dataset, get_aggregates
from utils.formatting import fmt_usd, fmt_usd_compact, fmt_pct, fmt_int
from components.charts import (
    revenue_profit_trend, category_revenue_profit, product_ranking_bars,
    discount_margin_scatter, customer_value_scatter,
)
from components.world_map import profit_world_map, shipping_world_map
print('imports ok')

imports ok


## 1. Load the dataset (once)
Uses the fast Parquet cache (`data/processed.parquet`) when present, otherwise converts the XLSX a single time.

In [2]:
df = load_dataset()
print(f'rows: {len(df):,} | columns: {df.shape[1]}')
print(f'period: {df["order_date"].min()} -> {df["order_date"].max()}')
print(f'countries: {sorted(df["country"].unique())}')
print(f'categories: {sorted(df["category"].unique())}')
df[["order_id", "country", "category", "total_price_usd", "profit_usd", "order_status", "delivery_status"]].head(3)

rows: 1,000,123 | columns: 37
period: 2024-02-03 04:31:57.621000 -> 2026-02-02 16:03:01.728000
countries: ['Australia', 'Belgium', 'Canada', 'France', 'Germany', 'Italy', 'Netherlands', 'Spain', 'United Kingdom', 'United States']
categories: ['Clothing', 'Electronics', 'Health', 'Home', 'Sports']


,order_id,country,category,total_price_usd,profit_usd,order_status,delivery_status
0,ORD-XAJI0,Netherlands,Home,325.38,93.24,Completed,In Transit
1,ORD-NHJ7X,United States,Health,102.56,34.78,Completed,Delivered
2,ORD-YTJXE,France,Home,887.67,468.45,Completed,In Transit


## 2. Executive KPIs

In [3]:
agg = get_aggregates(df)
t = agg['totals']
for k, v in [('Total Revenue', t['revenue']), ('Net Profit', t['net_profit']), ('Profit Leakage', t['leakage_total'])]:
    print(f'{k:15s} {fmt_usd(v)}')
print(f'{"Total Orders":15s} {fmt_int(t["orders"])}')
print(f'{"Avg Order Value":15s} {fmt_usd(t["aov"])}')
print(f'Leakage share of revenue: {fmt_pct(t["leakage_pct"])}')

Total Revenue   $403,285,857.00
Net Profit      $140,781,583.13
Profit Leakage  $97,707,231.21
Total Orders    991,930
Avg Order Value $406.57
Leakage share of revenue: 24.2%


## 3. Profit leakage breakdown

In [4]:
lk = agg['leakage']
pd.DataFrame([
    ('Discounts', lk['discounts'], lk['discounts_pct']),
    ('Returns / Refunds', lk['returns'], lk['returns_pct']),
    ('Shipping Costs', lk['shipping'], lk['shipping_pct']),
    ('Failed Deliveries (memo)', lk['failed'], lk['failed_pct']),
    ('Payment Fees (est.)', lk['fees'], lk['fees_pct']),
    ('TOTAL', lk['total'], lk['total_pct']),
], columns=['Source', 'Amount (USD)', '% of Revenue'])

,Source,Amount (USD),% of Revenue
0,Discounts,37481503.34,9.294029
1,Returns / Refunds,40060498.12,9.933524
2,Shipping Costs,12505985.29,3.101023
3,Failed Deliveries (memo),625127.95,0.155009
4,Payment Fees (est.),7659244.46,1.899210
5,TOTAL,97707231.21,24.227785


## 4. Revenue vs Profit trend

In [5]:
revenue_profit_trend(agg['trend']).show()

## 5. Profit by Country (world map)
Country fill = profit (pink = negative). Hover shows revenue, profit, margin, orders.

In [6]:
agg['country']
profit_world_map(agg['country']).show()

## 6. Global shipping & delivery analysis (by `shipping_country`)

In [7]:
agg['shipping']
shipping_world_map(agg['shipping']).show()

## 7. Customer value (segments from actual profit tertiles)

In [8]:
agg['customers'].groupby('segment').agg(
    customers=('customer_id', 'count'), revenue=('revenue', 'sum'), profit=('profit', 'sum'))
customer_value_scatter(agg['customers']).show()

## 8. Product analysis

In [9]:
agg['category']
category_revenue_profit(agg['category']).show()
product_ranking_bars(agg['products_ranked']).show()
print('Discount vs margin sample:')
discount_margin_scatter(agg['scatter']).show()

Discount vs margin sample:


## Conclusion

On the full dataset (1,000,123 rows): revenue **$403.29M**, net profit **$140.78M**, identified leakage **$97.71M (24.2% of revenue)** — led by returns ($40.06M) and discounts ($37.48M), then shipping ($12.51M) and estimated payment fees ($7.66M). The top customer-profit tertile (High segment) drives roughly two-thirds of total profit, and country/category breakdowns show where margin is won or lost. The interactive Dash dashboard (`python app.py`) presents these same computed results across four pages: Executive Overview, Profit Leakage, Customer Value, and Product Analysis.